In [1]:
## imports and functions for exploring the PAM Rodeo glider data

import pandas as pd
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import h5py
import os

def inspect_h5_structure(group, indent=0):
    """Print type/shape/dtype for every item in an h5py group, to see why a flat loop over .keys() doesn't work."""
    for name, item in group.items():
        if isinstance(item, h5py.Group):
            print(' ' * indent + f'{name}/  (Group, not a Dataset - cannot be sliced with [:])')
            inspect_h5_structure(item, indent + 4)
        else:
            print(' ' * indent + f'{name}: shape={item.shape}, dtype={item.dtype}')



# 2D level matrices (time x frequency bin) paired with the 1D frequency vector
# used to label their columns, instead of reading each as its own single column
FREQ_PAIRS = {
    'decadeLevels': 'decadeFreqHz',
    'hybridMiliDecLevels': 'hybridDecFreqHz',
    'thirdoct': 'thirdOctFreqHz',
}

def build_noise_df(pam_noise):
    """
    Build a DataFrame from the PAM noise HDF5 file.

    The datasets under GliderRodeo are not uniform, so a flat loop over
    .keys() doesn't work:
    - 'Parameters' is a Group, not a Dataset, so it can't be sliced with [:]
      and is skipped here.
    - 'DateTime' is an object array of variable-length byte strings and
      needs elementwise decoding.
    - 'decadeLevels' / 'hybridMiliDecLevels' / 'thirdoct' are 2D
      (time x frequency) matrices; their columns are labeled using the
      paired *FreqHz vector rather than being treated as one time series.
      'hybridDecFreqHz' has 3 columns (band edges + center) - the middle
      column is used as the representative frequency.
    - Everything else (e.g. 'broadband') is a single time-varying column.
    """
    glider_group = pam_noise['GliderRodeo']

    freq_vectors = {}
    for freq_name in FREQ_PAIRS.values():
        fv = glider_group[freq_name][:]
        if fv.ndim == 2 and fv.shape[1] == 1:
            fv = fv[:, 0]
        elif fv.ndim == 2:
            # multi-column band-edge table (e.g. [low, center, high]) -
            # use the middle column as the representative/center frequency
            fv = fv[:, fv.shape[1] // 2]
        freq_vectors[freq_name] = fv

    columns = {}
    for name, item in glider_group.items():
        if not isinstance(item, h5py.Dataset):
            continue  # e.g. 'Parameters' group

        if name in FREQ_PAIRS.values():
            continue  # only used to label columns of the matching *Levels matrix

        data = item[:]

        if data.dtype == object:  # h5py variable-length strings -> array of bytes/str objects
            data = np.array([x.decode('utf-8') if isinstance(x, bytes) else x for x in data])
        elif data.dtype.kind == 'S':  # fixed-length bytes -> string
            data = data.astype(str)

        if data.ndim == 2 and data.shape[1] == 1:
            data = data[:, 0]

        if name in FREQ_PAIRS:
            freqs = freq_vectors[FREQ_PAIRS[name]]
            for i, f in enumerate(freqs):
                columns[f'{name}_{f:g}Hz'] = data[:, i]
        elif data.ndim == 1:
            columns[name] = data
        else:
            # unexpected extra 2D dataset - fall back to indexed columns
            for i in range(data.shape[1]):
                columns[f'{name}_{i}'] = data[:, i]

    noise_df = pd.DataFrame(columns)

    if 'DateTime' in noise_df.columns:
        noise_df['DateTime'] = pd.to_datetime(noise_df['DateTime'], errors='coerce')

    return noise_df


In [7]:

glider_id = 'sg607'

print(f'Analyzing {glider_id}')
print('Ensure your paths are correct for your local environment!!\n')

## jupyter hub paths
glider_sci_path = f'/home/jovyan/shared-public/GliderRodeo/{glider_id}_20260128/{glider_id}_20260128_science_timeseries.csv'
pam_noise_path = f'/home/jovyan/shared-public/GliderRodeo/Noise/{glider_id}_20260128/{glider_id}_20260128.h5'
phase_key_path= '/home/jovyan/shared-public/GliderRodeo/glider_rodeo_phase_key.xlsx'


# local_paths

# glider_sci_path = f'./{glider_id}_20260128_science_timeseries.csv'
# pam_noise_path = f'./{glider_id}_20260128.h5'
# phase_key_path= './glider_rodeo_phase_key.xlsx'

print(glider_sci_path)
print(pam_noise_path)
print(phase_key_path)

print('###############################################################\n')
print('Reading glider science data from:', glider_sci_path)
glider_sci_df = pd.read_csv(glider_sci_path)
# 'time' is Unix epoch seconds (UTC) - confirmed against the PAM DateTime
# start (2026-01-28 22:23:58) and phase key deployment start (2026-01-28 22:16:00)
glider_sci_df['DateTime'] = pd.to_datetime(glider_sci_df['time'], unit='s')
print(glider_sci_df.head())
print('###############################################################\n')

print('Reading PAM noise data from:', pam_noise_path)
print('Inspecting PAM h5 structure:')
pam_noise = h5py.File(pam_noise_path, 'r')
inspect_h5_structure(pam_noise['GliderRodeo'])
pam_noise_df = build_noise_df(pam_noise)
print(pam_noise_df.head())
print('###############################################################\n')

print('Reading phase key data from:', phase_key_path)
phase_key_df = pd.read_excel(phase_key_path, sheet_name=glider_id.upper(), skiprows=1)
print(phase_key_df.head())
print('\n')

Analyzing sg607
Ensure your paths are correct for your local environment!!

/home/jovyan/shared-public/GliderRodeo/sg607_20260128/sg607_20260128_science_timeseries.csv
/home/jovyan/shared-public/GliderRodeo/Noise/sg607_20260128/sg607_20260128.h5
/home/jovyan/shared-public/GliderRodeo/glider_rodeo_phase_key.xlsx
###############################################################

Reading glider science data from: /home/jovyan/shared-public/GliderRodeo/sg607_20260128/sg607_20260128_science_timeseries.csv
   dive        time   latitude   longitude     depth  temperature  salinity  \
0     1  1769638886  21.212667 -158.157017  1.180368    25.500914       NaN   
1     1  1769638891  21.212667 -158.157017  1.170365    25.501297       NaN   
2     1  1769638896  21.212667 -158.157017  1.350420    25.479983       NaN   
3     1  1769638901  21.212667 -158.157017  1.230383    25.480549       NaN   
4     1  1769638906  21.212667 -158.157017  1.240386    25.470384       NaN   

   density  soundVelo

In [8]:
phase_key_df

,Phase,Mode,Start time (UTC),Stop time (UTC),Start Dive Number,Stop Dive Number,MAX_BUOY,HEAD_ERRBAND,GC timing,target w (cm/s),T_DIVE (for 990 m)
0,1.0,shakedown,2026-01-28 22:16:00,2026-01-30 01:44:00,1,11,NaN,NaN,NaN,NaN,NaN
1,2.0,fast,2026-01-30 02:42:00,2026-02-01 18:50:00,12,27,250,10,120 s all depths,11,300
2,NaN,shallow dives,2026-01-31 08:04:00,2026-01-31 16:47:00,18,21,-,-,-,-,-
3,3.0,slow,2026-02-01 18:58:00,2026-02-04 17:34:00,28,39,120,30,300 s below 150 m,9.167,360
4,4.0,intermediate,2026-02-04 18:07:00,2026-02-06 18:34:00,40,48,150,NaN,"60 s to 50 m, 120 s to 150 m, 300 s",10,330
5,5.0,drift,2026-02-06 18:58:00,2026-02-08 20:40:00,49,58,150,NaN,"60 s to 50 m, 120 s to 150 m, 300 s",10,330
6,6.0,mini-rodeo,-,-,-,-,-,-,-,-,-
7,NaN,mini-slow,2026-02-08 20:49:00,2026-02-09 08:21:00,59,60,120,30,300 s below 150 m,9.167,360
8,NaN,mini-intermediate,2026-02-09 08:41:00,2026-02-09 19:19:00,61,62,150,NaN,"60 s to 50 m, 120 s to 150 m, 300 s",10,330
9,NaN,mini-fast,2026-02-09 20:15:00,2026-02-10 06:45:00,63,64,250,10,120 s all depths,11,300


In [9]:
# PAM noise columns to bring onto glider_sci_df - add more here as needed (must be column names in pam_noise_df)
noise_vars_to_merge = ['broadband']
print(f'Noise columns to merge: {noise_vars_to_merge}')
# max allowed gap between a science row's timestamp and the nearest PAM sample -
# beyond this, the merged noise values are NaN instead of silently reusing a
# distant/stale PAM value (e.g. before PAM logging started, or during a gap)
noise_merge_tolerance = pd.Timedelta('90s')
print(f'Merging PAM noise columns {noise_vars_to_merge} onto glider science data with tolerance {noise_merge_tolerance}...')

# drop first so re-running this cell (without restarting the kernel) doesn't collide with
# the previous run's merged columns and get silently renamed to *_x/*_y
glider_sci_df = glider_sci_df.drop(columns=noise_vars_to_merge, errors='ignore')

noise_subset = pam_noise_df[['DateTime'] + noise_vars_to_merge].sort_values('DateTime')
glider_sci_df = glider_sci_df.sort_values('DateTime')

glider_sci_df = pd.merge_asof(
    glider_sci_df,
    noise_subset,
    on='DateTime',
    direction='nearest',
    tolerance=noise_merge_tolerance
)

glider_sci_df.head()

Noise columns to merge: ['broadband']
Merging PAM noise columns ['broadband'] onto glider science data with tolerance 0 days 00:01:30...


,dive,time,latitude,longitude,depth,temperature,salinity,density,soundVelocity,DateTime,broadband
0,1,1769638886,21.212667,-158.157017,1.180368,25.500914,NaN,NaN,NaN,2026-01-28 22:21:26,NaN
1,1,1769638891,21.212667,-158.157017,1.170365,25.501297,NaN,NaN,NaN,2026-01-28 22:21:31,NaN
2,1,1769638896,21.212667,-158.157017,1.350420,25.479983,NaN,NaN,NaN,2026-01-28 22:21:36,NaN
3,1,1769638901,21.212667,-158.157017,1.230383,25.480549,NaN,NaN,NaN,2026-01-28 22:21:41,NaN
4,1,1769638906,21.212667,-158.157017,1.240386,25.470384,NaN,NaN,NaN,2026-01-28 22:21:46,NaN


In [10]:
# Assign a repeating 'phase' label to glider_sci_df based on which dive number
# range each phase_key_df row covers (Mode is the phase label, e.g. 'shakedown', 'fast')
glider_sci_df['phase'] = pd.NA

for _, row in phase_key_df.iterrows():
    start_dive = pd.to_numeric(row['Start Dive Number'], errors='coerce')
    stop_dive = pd.to_numeric(row['Stop Dive Number'], errors='coerce')

    if pd.isna(start_dive):
        continue  # no numeric start - can't place this phase on the dive axis (e.g. '-' for mini-rodeo)
    print(f'Processing phase: {row["Mode"]}, Dive range: {start_dive} to {stop_dive}')
    if pd.isna(stop_dive):
        # non-numeric stop (e.g. 'recovery' row's stop is 'WHICEAS', a location not a dive number) -
        # treat as open-ended through the last dive in the data
        stop_dive = glider_sci_df['dive'].max()

    mask = glider_sci_df['dive'].between(start_dive, stop_dive)
    glider_sci_df.loc[mask, 'phase'] = row['Mode']

glider_sci_df[['dive', 'phase']].drop_duplicates()

Processing phase: shakedown, Dive range: 1 to 11
Processing phase: fast, Dive range: 12 to 27
Processing phase: shallow dives, Dive range: 18 to 21
Processing phase: slow, Dive range: 28 to 39
Processing phase: intermediate, Dive range: 40 to 48
Processing phase: drift, Dive range: 49 to 58
Processing phase: mini-slow, Dive range: 59 to 60
Processing phase: mini-intermediate, Dive range: 61 to 62
Processing phase: mini-fast, Dive range: 63 to 64
Processing phase: recovery, Dive range: 65 to nan


,dive,phase
0,1,shakedown
127,2,shakedown
382,3,shakedown
741,4,shakedown
1164,5,shakedown
...,...,...
119090,61,mini-intermediate
121357,62,mini-intermediate
123663,63,mini-fast
125750,64,mini-fast


In [11]:
pam_noise.keys()

<KeysViewHDF5 ['GliderRodeo']>

In [12]:
pam_noise['GliderRodeo'].keys()

<KeysViewHDF5 ['DateTime', 'Parameters', 'broadband', 'decadeFreqHz', 'decadeLevels', 'hybridDecFreqHz', 'hybridMiliDecLevels', 'thirdOctFreqHz', 'thirdoct']>

In [13]:
pam_noise['GliderRodeo']['Parameters']

<HDF5 group "/GliderRodeo/Parameters" (0 members)>

## QC

TODO: add QC checks here (e.g. flag NaN gaps in merged broadband, out-of-range CTD values,
merge-tolerance misses) before exporting for stats/plots.

In [14]:
# placeholder - QC checks go here once defined

## Export

- Everything is saved under a per-glider folder (`./noise_analysis_outputs/{glider_id}/`), which
  the stats/plots and presentation notebooks also read from/write to - so multiple gliders never
  collide or overwrite each other's outputs.
- `glider_sci_df` (merged science + broadband SPL + phase) -> CSV for the stats/plots notebook.
- The full hybrid millidecade spectrum (2797 bands, `pam_noise_df`) -> parquet, since it's too
  large for a practical CSV (2797 cols x ~15876 rows). Only `DateTime` + the hybrid bands are
  exported here since that's the only spectrum resolution the stats/plots notebook uses.

In [15]:


# one output folder per glider - everything else (stats/plots notebook, presentation notebook)
# reads from and writes to this same folder
OUT_DIR = f'./noise_analysis_outputs/{glider_id}'
os.makedirs(OUT_DIR, exist_ok=True)

glider_export_path = os.path.join(OUT_DIR, f'{glider_id}_merged_science.csv')
glider_sci_df.to_csv(glider_export_path, index=False)
print(f'Saved merged science data to {glider_export_path} ({len(glider_sci_df):,} rows)')

hybrid_cols = [c for c in pam_noise_df.columns if c.startswith('hybridMiliDecLevels_')]
spectrum_export_path = os.path.join(OUT_DIR, f'{glider_id}_pam_spectrum.parquet')
pam_noise_df[['DateTime'] + hybrid_cols].to_parquet(spectrum_export_path, index=False)
print(f'Saved PAM hybrid millidecade spectrum to {spectrum_export_path} '
      f'({len(hybrid_cols)} bands x {len(pam_noise_df):,} samples)')

Saved merged science data to ./noise_analysis_outputs/sg607/sg607_merged_science.csv (128,395 rows)
Saved PAM hybrid millidecade spectrum to ./noise_analysis_outputs/sg607/sg607_pam_spectrum.parquet (2797 bands x 15,876 samples)
